# Clipt Detection Models — v3 OCR Notebook
# ALI REPLACEMENT LAYER — 12 Models, No Ensemble
#
# IMPORTANT: Every model has its own download cell.
# Run the download cell immediately after each training 
# cell finishes. Do NOT wait for the full chunk.
# This protects your progress if Colab disconnects.
#
# INSTRUCTIONS:
# 1. Runtime → Change runtime type → A100 GPU
# 2. Add ROBOFLOW_API_KEY to Colab Secrets
# 3. Run setup cell first
# 4. For each model: run dataset cell → train cell → 
#    DOWNLOAD cell immediately
# 5. Total time: ~5-6 hours on A100
#
# MODEL PIPELINE (when all trained):
# player_isolator_v3 → jersey_color_classifier_v3 →
# number_region_detector_v3 → jersey_ocr_v3_primary →
# sport-specific model → specialist models →
# temporal_consensus (3+ frame agreement)
#
# FULL DETECTION LAYER ORDER:
# Step 1: Ali's ensemble (v1) — runs first, highest accuracy
# Step 2: jersey_number_universal_v1 (v2, mAP50 0.995) — best universal
# Step 3: v3 OCR pipeline (these models) — Ali replacement
# Step 4: v2 sport-specific models — parallel confirmation
# Step 5: temporal_consensus — 3+ frame filter
# Step 6: Stat generation (zones, actions, ball tracking)
#
# Model Filename Cross-Reference (MUST match roboflow_detector.py):
# CHUNK 1 — Multi-sport Jersey OCR
#   jersey_ocr_v3_primary.pt      (13,815 images)
#   jersey_ocr_v3_secondary.pt    (6,932 images)
#
# CHUNK 2 — Sport-specific OCR
#   basketball_ocr_v3.pt          (826 images)
#   football_ocr_v3.pt            (2,918 images)
#   lacrosse_ocr_v3.pt            (2,100 images)
#
# CHUNK 3 — Player Isolation + Color
#   player_isolator_v3.pt         (5,174 images)
#   jersey_color_classifier_v3.pt (1,232 images)
#   number_region_detector_v3.pt  (556 images)
#
# CHUNK 4 — Augmentation Specialists
#   motion_blur_specialist_v3.pt
#   wide_angle_specialist_v3.pt   (imgsz=1280)
#   dark_jersey_specialist_v3.pt
#   partial_visibility_specialist_v3.pt

# ⚠️ IF COLAB DISCONNECTS:
# 1. Go to colab.research.google.com
# 2. Click Recent → find train_models_v3.ipynb
# 3. Runtime → Connect to hosted runtime: A100
# 4. If reconnect fails → Runtime → Run all WON'T work
#    Instead: rerun Setup cell only, then rerun 
#    dataset download cells for the current chunk only,
#    then continue from the last incomplete training cell
# 5. Already downloaded models are safe in your Downloads
#    folder — you never need to retrain those

## Setup — Run this first (every session)

In [ ]:
import os

# ── GPU verification ───────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "No GPU — go to Runtime → Change runtime type → A100"
device_name = torch.cuda.get_device_name(0)
print(f"GPU: {device_name}")

# ── Read API key from Colab Secrets (key icon in sidebar) ─────
from google.colab import userdata
api_key = userdata.get('ROBOFLOW_API_KEY')

!pip install roboflow ultralytics -q
from roboflow import Roboflow
from ultralytics import YOLO

rf = Roboflow(api_key=api_key)

# ── v3 training constants ─────────────────────────────────────────────────
V3_BASE = "yolov8m.pt"   # 25.9M params — medium backbone for OCR accuracy
V3_IMGSZ = 832            # Higher res for small text OCR
V3_BATCH = 8              # Larger model = smaller batch
V3_DEVICE = 0

print(f"Setup complete — GPU: {device_name}, base: {V3_BASE}, imgsz: {V3_IMGSZ}")

# ================================================================
# CHUNK 1 — Multi-sport OCR (Primary Ali Replacement)
# ================================================================
# 
# These models run AFTER Ali's ensemble fails (returns 0 detections)
# and ALONGSIDE jersey_number_universal_v1 (v2, mAP50 0.995)
# 
# jersey_ocr_v3_primary (13,815 images, YOLOv8m):
#   - Primary replacement for Ali's OCR
#   - Runs on ALL sports
#   - If this + universal_v1 agree → high confidence detection
#
# jersey_ocr_v3_secondary (6,932 images, YOLOv8m):
#   - Second opinion alongside primary
#   - Agreement between primary + secondary + universal_v1
#     = extremely high confidence detection
#
# 2 models × ~25 min each = ~50 min on A100

In [ ]:
# ── Download multi-sport OCR datasets ─────────────────────

print("Downloading multi-sport OCR datasets...")

# Primary: largest digit-level jersey number dataset available
# 13,815 images, 1 class (digit), MIT license
print("\n1/2 Primary — digit detection (13,815 images):")
try:
    project = rf.workspace("footballplayertracking").project("jerseynumberdetectordigitdetector")
    dataset_primary = project.version(1).download("yolov8")
    print(f"✅ Downloaded: {dataset_primary.location}")
except Exception as e:
    print(f"Version 1 failed: {e}")
    try:
        dataset_primary = project.version(0).download("yolov8")
        print(f"✅ Downloaded v0: {dataset_primary.location}")
    except Exception as e2:
        print(f"❌ Both versions failed — skipping")
        dataset_primary = None

# Secondary: multi-class jersey number detection
# MUST be a DIFFERENT dataset from primary to provide unique training data.
# Primary source: volleyai (6,932 images, 12 classes)
# Fallback: dark-blue-jt0mg/jerseynumbers (826 images, 10 digit classes)
print("\n2/2 Secondary — jersey number detection (6,932 images):")
dataset_secondary = None
try:
    project = rf.workspace("volleyai-actions").project("jersey-number-detection-s01j4")
    dataset_secondary = project.version(2).download("yolov8")
    print(f"✅ Secondary dataset: {dataset_secondary.location}")
except Exception as e:
    print(f"v2 failed: {e}")
    try:
        dataset_secondary = project.version(1).download("yolov8")
        print(f"✅ Secondary v1: {dataset_secondary.location}")
    except Exception as e2:
        print(f"❌ volleyai download failed: {e2}")
        # Fallback: completely different dataset (826 images, NOT primary)
        print("Falling back to dark-blue-jt0mg/jerseynumbers (826 images)...")
        try:
            project = rf.workspace("dark-blue-jt0mg").project("jerseynumbers")
            dataset_secondary = project.version(5).download("yolov8")
            print(f"✅ Fallback secondary: {dataset_secondary.location}")
        except Exception as e3:
            print(f"❌ Fallback also failed: {e3}")
            dataset_secondary = None

# Safety check: secondary must NEVER point to the same dataset as primary
if dataset_primary is not None and dataset_secondary is not None:
    if dataset_secondary.location == dataset_primary.location:
        print("❌ CRITICAL: Secondary resolved to same path as primary — clearing secondary")
        dataset_secondary = None

ready = sum(1 for d in [dataset_primary, dataset_secondary] if d is not None)
print(f"\n{'✅' if ready == 2 else '⚠️'} {ready}/2 Chunk 1 datasets ready")

In [ ]:
# ── Train jersey_ocr_v3_primary ───────────────────────────────
# 13,815 images — LARGE tier (epochs=75)
if dataset_primary is not None:
    print("=" * 60)
    print("TRAINING: jersey_ocr_v3_primary (13,815 images, YOLOv8m)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_primary.location}/data.yaml",
        epochs=75,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="jersey_ocr_v3_primary",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.8,
        hsv_v=0.5,
        degrees=15,
        translate=0.15,
        scale=0.7,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.1,
    )
    print("✅ Training complete: jersey_ocr_v3_primary")
else:
    print("⏭️ SKIPPED: jersey_ocr_v3_primary — dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🔽 IMMEDIATE DOWNLOAD — jersey_ocr_v3_primary
# Run this RIGHT AFTER training completes!
# ════════════════════════════════════════════════════════════════
from google.colab import files
import shutil, os

model_name = "jersey_ocr_v3_primary.pt"
run_name = "jersey_ocr_v3_primary"
path = f"runs/detect/{run_name}/weights/best.pt"

if os.path.exists(path):
    from ultralytics import YOLO
    metrics = YOLO(path).val()
    map50 = metrics.box.map50
    size_mb = os.path.getsize(path) / 1024 / 1024
    if map50 >= 0.5:
        shutil.copy(path, model_name)
        files.download(model_name)
        print(f"✅ DOWNLOADED: {model_name} mAP50={map50:.3f} ({size_mb:.1f}MB)")
    else:
        print(f"❌ FAILED: {model_name} mAP50={map50:.3f} — below 0.5 threshold, not downloading")
else:
    print(f"❌ MISSING: {path} — training may not have completed")

In [ ]:
# ── Train jersey_ocr_v3_secondary ─────────────────────────────
# 6,932 images — LARGE tier (epochs=75)

# GUARD: Stop immediately if secondary accidentally points to primary dataset.
# This prevents wasting 6+ hours training on the wrong (duplicate) data.
if dataset_secondary is not None and dataset_primary is not None:
    assert dataset_secondary.location != dataset_primary.location, \
        "ERROR: Secondary is pointing to primary dataset — fix download cell first"

if dataset_secondary is not None:
    print("=" * 60)
    print("TRAINING: jersey_ocr_v3_secondary (6,932 images, YOLOv8m)")
    print(f"Dataset: {dataset_secondary.location}")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_secondary.location}/data.yaml",
        epochs=75,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="jersey_ocr_v3_secondary",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.8,
        hsv_v=0.5,
        degrees=15,
        translate=0.15,
        scale=0.7,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.1,
    )
    print("✅ Training complete: jersey_ocr_v3_secondary")
else:
    print("⏭️ SKIPPED: jersey_ocr_v3_secondary — dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🔽 IMMEDIATE DOWNLOAD — jersey_ocr_v3_secondary
# Run this RIGHT AFTER training completes!
# ════════════════════════════════════════════════════════════════
from google.colab import files
import shutil, os

model_name = "jersey_ocr_v3_secondary.pt"
run_name = "jersey_ocr_v3_secondary"
path = f"runs/detect/{run_name}/weights/best.pt"

if os.path.exists(path):
    from ultralytics import YOLO
    metrics = YOLO(path).val()
    map50 = metrics.box.map50
    size_mb = os.path.getsize(path) / 1024 / 1024
    if map50 >= 0.5:
        shutil.copy(path, model_name)
        files.download(model_name)
        print(f"✅ DOWNLOADED: {model_name} mAP50={map50:.3f} ({size_mb:.1f}MB)")
    else:
        print(f"❌ FAILED: {model_name} mAP50={map50:.3f} — below 0.5 threshold, not downloading")
else:
    print(f"❌ MISSING: {path} — training may not have completed")

# ================================================================
# CHUNK 2 — Sport-specific OCR (~75 min on A100)
# ================================================================
#
# These models run AFTER the primary v3 OCR models.
# They provide sport-specific digit pattern expertise:
#   basketball: larger digits, high contrast jerseys
#   football: motion blur, dark navy jerseys (Dustin's #11)
#   lacrosse: general number detection (no lacrosse OCR dataset exists)
#
# In the pipeline these run as part of Step 3 (v3 OCR pipeline),
# called by roboflow_detector._run_v3_ocr_on_crop() based on sport.
#
# Cross-validation: if sport-specific + primary agree → +0.15 confidence
#
# 3 models × ~25 min each = ~75 min on A100

In [ ]:
# ── Download sport-specific OCR datasets ──────────────────

print("Downloading sport-specific OCR datasets...")

# Basketball: 10 digit classes (0-9), 826 images
print("\n1/3 Basketball digits (826 images, 10 digit classes):")
try:
    project = rf.workspace("dark-blue-jt0mg").project("jerseynumbers")
    dataset_bball_ocr = project.version(5).download("yolov8")
    print(f"✅ Downloaded: {dataset_bball_ocr.location}")
except Exception as e:
    print(f"Version 5 failed: {e}")
    try:
        dataset_bball_ocr = project.version(4).download("yolov8")
        print(f"✅ Downloaded v4: {dataset_bball_ocr.location}")
    except Exception as e2:
        print(f"❌ Both versions failed — skipping")
        dataset_bball_ocr = None

# Football: jersey tracker, 2,918 images
print("\n2/3 Football jersey tracker (2,918 images):")
try:
    project = rf.workspace("football-tracking").project("football-jersey-tracker")
    dataset_fb_ocr = project.version(1).download("yolov8")
    print(f"✅ Downloaded: {dataset_fb_ocr.location}")
except Exception as e:
    print(f"Version 1 failed: {e}")
    try:
        dataset_fb_ocr = project.version(0).download("yolov8")
        print(f"✅ Downloaded v0: {dataset_fb_ocr.location}")
    except Exception as e2:
        print(f"❌ Both versions failed — skipping")
        dataset_fb_ocr = None

# Lacrosse: general number detection (~2,100 images)
print("\n3/3 Lacrosse/general number detection (2,100 images):")
try:
    project = rf.workspace("smart-scoreboard").project("number-rr9yl-nd3hg")
    dataset_lax_ocr = project.version(1).download("yolov8")
    print(f"✅ Downloaded: {dataset_lax_ocr.location}")
except Exception as e:
    print(f"Version 1 failed: {e}")
    try:
        dataset_lax_ocr = project.version(0).download("yolov8")
        print(f"✅ Downloaded v0: {dataset_lax_ocr.location}")
    except Exception as e2:
        print(f"❌ Both versions failed — skipping")
        dataset_lax_ocr = None

ready = sum(1 for d in [dataset_bball_ocr, dataset_fb_ocr, dataset_lax_ocr] if d is not None)
print(f"\n{'✅' if ready == 3 else '⚠️'} {ready}/3 Chunk 2 datasets ready")

In [ ]:
# ── Train basketball_ocr_v3 ──────────────────────────────────
# 826 images — MEDIUM tier (epochs=100)
if dataset_bball_ocr is not None:
    print("=" * 60)
    print("TRAINING: basketball_ocr_v3 (826 images, YOLOv8m)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_bball_ocr.location}/data.yaml",
        epochs=100,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="basketball_ocr_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.9,
        hsv_v=0.5,
        degrees=20,
        translate=0.2,
        scale=0.7,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.15,
        copy_paste=0.15,
    )
    print("✅ Training complete: basketball_ocr_v3")
else:
    print("⏭️ SKIPPED: basketball_ocr_v3 — dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🔽 IMMEDIATE DOWNLOAD — basketball_ocr_v3
# Run this RIGHT AFTER training completes!
# ════════════════════════════════════════════════════════════════
from google.colab import files
import shutil, os

model_name = "basketball_ocr_v3.pt"
run_name = "basketball_ocr_v3"
path = f"runs/detect/{run_name}/weights/best.pt"

if os.path.exists(path):
    from ultralytics import YOLO
    metrics = YOLO(path).val()
    map50 = metrics.box.map50
    size_mb = os.path.getsize(path) / 1024 / 1024
    if map50 >= 0.5:
        shutil.copy(path, model_name)
        files.download(model_name)
        print(f"✅ DOWNLOADED: {model_name} mAP50={map50:.3f} ({size_mb:.1f}MB)")
    else:
        print(f"❌ FAILED: {model_name} mAP50={map50:.3f} — below 0.5 threshold, not downloading")
else:
    print(f"❌ MISSING: {path} — training may not have completed")

In [ ]:
# ── Train football_ocr_v3 ────────────────────────────────────
# 2,918 images — LARGE tier (epochs=100)
if dataset_fb_ocr is not None:
    print("=" * 60)
    print("TRAINING: football_ocr_v3 (2,918 images, YOLOv8m)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_fb_ocr.location}/data.yaml",
        epochs=100,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="football_ocr_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.9,
        hsv_v=0.5,
        degrees=20,
        translate=0.2,
        scale=0.7,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.15,
        copy_paste=0.15,
    )
    print("✅ Training complete: football_ocr_v3")
else:
    print("⏭️ SKIPPED: football_ocr_v3 — dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🔽 IMMEDIATE DOWNLOAD — football_ocr_v3
# Run this RIGHT AFTER training completes!
# ════════════════════════════════════════════════════════════════
from google.colab import files
import shutil, os

model_name = "football_ocr_v3.pt"
run_name = "football_ocr_v3"
path = f"runs/detect/{run_name}/weights/best.pt"

if os.path.exists(path):
    from ultralytics import YOLO
    metrics = YOLO(path).val()
    map50 = metrics.box.map50
    size_mb = os.path.getsize(path) / 1024 / 1024
    if map50 >= 0.5:
        shutil.copy(path, model_name)
        files.download(model_name)
        print(f"✅ DOWNLOADED: {model_name} mAP50={map50:.3f} ({size_mb:.1f}MB)")
    else:
        print(f"❌ FAILED: {model_name} mAP50={map50:.3f} — below 0.5 threshold, not downloading")
else:
    print(f"❌ MISSING: {path} — training may not have completed")

In [ ]:
# ── Train lacrosse_ocr_v3 ────────────────────────────────────
# 2,100 images — MEDIUM tier (epochs=100)
if dataset_lax_ocr is not None:
    print("=" * 60)
    print("TRAINING: lacrosse_ocr_v3 (2,100 images, YOLOv8m)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_lax_ocr.location}/data.yaml",
        epochs=100,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="lacrosse_ocr_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.9,
        hsv_v=0.5,
        degrees=20,
        translate=0.2,
        scale=0.7,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.15,
        copy_paste=0.15,
    )
    print("✅ Training complete: lacrosse_ocr_v3")
else:
    print("⏭️ SKIPPED: lacrosse_ocr_v3 — dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🔽 IMMEDIATE DOWNLOAD — lacrosse_ocr_v3
# Run this RIGHT AFTER training completes!
# ════════════════════════════════════════════════════════════════
from google.colab import files
import shutil, os

model_name = "lacrosse_ocr_v3.pt"
run_name = "lacrosse_ocr_v3"
path = f"runs/detect/{run_name}/weights/best.pt"

if os.path.exists(path):
    from ultralytics import YOLO
    metrics = YOLO(path).val()
    map50 = metrics.box.map50
    size_mb = os.path.getsize(path) / 1024 / 1024
    if map50 >= 0.5:
        shutil.copy(path, model_name)
        files.download(model_name)
        print(f"✅ DOWNLOADED: {model_name} mAP50={map50:.3f} ({size_mb:.1f}MB)")
    else:
        print(f"❌ FAILED: {model_name} mAP50={map50:.3f} — below 0.5 threshold, not downloading")
else:
    print(f"❌ MISSING: {path} — training may not have completed")

# ================================================================
# CHUNK 3 — Player Isolation + Color (~60 min on A100)
# ================================================================
#
# These models are the FIRST step in the v3 OCR pipeline:
# 1. player_isolator_v3 finds player bounding boxes
#    → replaces football_player_detector (v1) as primary player finder
#    → falls back to v1 if v3 not loaded (graceful degradation)
#
# 2. jersey_color_classifier_v3 confirms correct player by color
#    → only process players whose jersey matches target color
#    → prevents wasting OCR on wrong team's players
#
# 3. number_region_detector_v3 finds exact number location
#    → crops just the number area for OCR models
#    → dramatically improves OCR accuracy on small numbers
#
# In the pipeline (Step 3):
# player_isolator_v3 → jersey_color_classifier_v3 →
# number_region_detector_v3 → OCR models from Chunk 1 + 2
#
# 3 models × ~20 min each = ~60 min on A100

In [ ]:
# ── Download player isolation datasets ────────────────────

print("Downloading player isolation + color datasets...")

# Player isolation: 5,174 images, player bounding boxes with team labels
print("\n1/3 Player isolator (5,174 images):")
try:
    project = rf.workspace("ai-in-sports").project("football-player-identification")
    dataset_player = project.version(1).download("yolov8")
    print(f"✅ Downloaded: {dataset_player.location}")
except Exception as e:
    print(f"Version 1 failed: {e}")
    try:
        dataset_player = project.version(0).download("yolov8")
        print(f"✅ Downloaded v0: {dataset_player.location}")
    except Exception as e2:
        print(f"❌ Both versions failed — skipping")
        dataset_player = None

# Jersey color: team color classification from player crops
# 1,232 images with team-color annotated bounding boxes
print("\n2/3 Jersey color classifier (1,232 images):")
try:
    project = rf.workspace("augmented-startups").project("football-player-detection-kucab")
    dataset_color = project.version(1).download("yolov8")
    print(f"✅ Downloaded: {dataset_color.location}")
except Exception as e:
    print(f"Version 1 failed: {e}")
    try:
        dataset_color = project.version(0).download("yolov8")
        print(f"✅ Downloaded v0: {dataset_color.location}")
    except Exception as e2:
        print(f"❌ Both versions failed — skipping")
        dataset_color = None

# Number region: 556 images, detects WHERE the number is on jersey
print("\n3/3 Number region detector (556 images):")
try:
    project = rf.workspace("yakovk").project("jersey-numbers-i1wn5")
    dataset_numregion = project.version(1).download("yolov8")
    print(f"✅ Downloaded: {dataset_numregion.location}")
except Exception as e:
    print(f"Version 1 failed: {e}")
    try:
        dataset_numregion = project.version(0).download("yolov8")
        print(f"✅ Downloaded v0: {dataset_numregion.location}")
    except Exception as e2:
        print(f"❌ Both versions failed — skipping")
        dataset_numregion = None

ready = sum(1 for d in [dataset_player, dataset_color, dataset_numregion] if d is not None)
print(f"\n{'✅' if ready == 3 else '⚠️'} {ready}/3 Chunk 3 datasets ready")

In [ ]:
# ── Train player_isolator_v3 ─────────────────────────────────
# 5,174 images — LARGE tier (epochs=75)
if dataset_player is not None:
    print("=" * 60)
    print("TRAINING: player_isolator_v3 (5,174 images, YOLOv8m)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_player.location}/data.yaml",
        epochs=75,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="player_isolator_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.8,
        hsv_v=0.5,
        degrees=15,
        translate=0.15,
        scale=0.7,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.1,
    )
    print("✅ Training complete: player_isolator_v3")
else:
    print("⏭️ SKIPPED: player_isolator_v3 — dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🔽 IMMEDIATE DOWNLOAD — player_isolator_v3
# Run this RIGHT AFTER training completes!
# ════════════════════════════════════════════════════════════════
from google.colab import files
import shutil, os

model_name = "player_isolator_v3.pt"
run_name = "player_isolator_v3"
path = f"runs/detect/{run_name}/weights/best.pt"

if os.path.exists(path):
    from ultralytics import YOLO
    metrics = YOLO(path).val()
    map50 = metrics.box.map50
    size_mb = os.path.getsize(path) / 1024 / 1024
    if map50 >= 0.5:
        shutil.copy(path, model_name)
        files.download(model_name)
        print(f"✅ DOWNLOADED: {model_name} mAP50={map50:.3f} ({size_mb:.1f}MB)")
    else:
        print(f"❌ FAILED: {model_name} mAP50={map50:.3f} — below 0.5 threshold, not downloading")
else:
    print(f"❌ MISSING: {path} — training may not have completed")

In [ ]:
# ── Train jersey_color_classifier_v3 ───────────────────────────
# 1,232 images — MEDIUM tier (epochs=100)
if dataset_color is not None:
    print("=" * 60)
    print("TRAINING: jersey_color_classifier_v3 (1,232 images, YOLOv8m)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_color.location}/data.yaml",
        epochs=100,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="jersey_color_classifier_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.9,
        hsv_v=0.5,
        degrees=20,
        translate=0.2,
        scale=0.7,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.15,
        copy_paste=0.15,
    )
    print("✅ Training complete: jersey_color_classifier_v3")
else:
    print("⏭️ SKIPPED: jersey_color_classifier_v3 — dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🔽 IMMEDIATE DOWNLOAD — jersey_color_classifier_v3
# Run this RIGHT AFTER training completes!
# ════════════════════════════════════════════════════════════════
from google.colab import files
import shutil, os

model_name = "jersey_color_classifier_v3.pt"
run_name = "jersey_color_classifier_v3"
path = f"runs/detect/{run_name}/weights/best.pt"

if os.path.exists(path):
    from ultralytics import YOLO
    metrics = YOLO(path).val()
    map50 = metrics.box.map50
    size_mb = os.path.getsize(path) / 1024 / 1024
    if map50 >= 0.5:
        shutil.copy(path, model_name)
        files.download(model_name)
        print(f"✅ DOWNLOADED: {model_name} mAP50={map50:.3f} ({size_mb:.1f}MB)")
    else:
        print(f"❌ FAILED: {model_name} mAP50={map50:.3f} — below 0.5 threshold, not downloading")
else:
    print(f"❌ MISSING: {path} — training may not have completed")

In [ ]:
# ── Train number_region_detector_v3 ────────────────────────────
# 556 images — MEDIUM tier (epochs=100)
if dataset_numregion is not None:
    print("=" * 60)
    print("TRAINING: number_region_detector_v3 (556 images, YOLOv8m)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_numregion.location}/data.yaml",
        epochs=100,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="number_region_detector_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.9,
        hsv_v=0.5,
        degrees=20,
        translate=0.2,
        scale=0.7,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.15,
        copy_paste=0.15,
    )
    print("✅ Training complete: number_region_detector_v3")
else:
    print("⏭️ SKIPPED: number_region_detector_v3 — dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🔽 IMMEDIATE DOWNLOAD — number_region_detector_v3
# Run this RIGHT AFTER training completes!
# ════════════════════════════════════════════════════════════════
from google.colab import files
import shutil, os

model_name = "number_region_detector_v3.pt"
run_name = "number_region_detector_v3"
path = f"runs/detect/{run_name}/weights/best.pt"

if os.path.exists(path):
    from ultralytics import YOLO
    metrics = YOLO(path).val()
    map50 = metrics.box.map50
    size_mb = os.path.getsize(path) / 1024 / 1024
    if map50 >= 0.5:
        shutil.copy(path, model_name)
        files.download(model_name)
        print(f"✅ DOWNLOADED: {model_name} mAP50={map50:.3f} ({size_mb:.1f}MB)")
    else:
        print(f"❌ FAILED: {model_name} mAP50={map50:.3f} — below 0.5 threshold, not downloading")
else:
    print(f"❌ MISSING: {path} — training may not have completed")

# ================================================================
# CHUNK 4 — Specialist Models (Hard Case Handlers) (~60 min on A100)
# ================================================================
#
# These run specifically when standard OCR fails on hard cases:
# - motion_blur_specialist: fast camera pans, running players
# - wide_angle_specialist: broadcast cameras, far-away shots
#   (common in high school game film like Dustin's St Mark's footage)
# - dark_jersey_specialist: for navy/black jerseys
#   (Dustin's football jersey #11 is NAVY — this model is critical)
# - partial_visibility: helmet/player blocking jersey number
#
# All specialists reuse the primary dataset (13,815 images)
# with EXTREME augmentation to simulate hard conditions.
#
# In the pipeline (Step 3):
# Standard OCR fails → specialists activated →
# dark_jersey + wide_angle most important for football
#
# PRIMARY TEST CASE:
# YouTube: XRSrRPbZIF0, jersey #11, navy, QB, St Mark's
# dark_jersey_specialist_v3 + wide_angle_specialist_v3
# are specifically designed for this exact video.
#
# 4 models × ~15 min each = ~60 min on A100

In [ ]:
# ── Re-download primary dataset if needed ─────────────────
# Chunk 4 reuses the primary dataset from Chunk 1.
# If Colab disconnected between chunks, re-download it.

if 'dataset_primary' not in dir() or dataset_primary is None:
    print("Re-downloading primary dataset for specialist training...")
    try:
        project = rf.workspace("footballplayertracking").project("jerseynumberdetectordigitdetector")
        dataset_primary = project.version(1).download("yolov8")
        print(f"✅ Downloaded: {dataset_primary.location}")
    except Exception as e:
        print(f"Version 1 failed: {e}")
        try:
            dataset_primary = project.version(0).download("yolov8")
            print(f"✅ Downloaded v0: {dataset_primary.location}")
        except Exception as e2:
            print(f"❌ Both versions failed")
            dataset_primary = None
else:
    print(f"✅ Primary dataset already loaded: {dataset_primary.location}")

if dataset_primary is None:
    print("❌ CRITICAL: Primary dataset unavailable — Chunk 4 cannot train.")
else:
    print("✅ Primary dataset ready for specialist training")

In [ ]:
# ── Train motion_blur_specialist_v3 ────────────────────────────
# Reuses primary dataset (13,815 images) — epochs=150, extreme blur augmentation
if dataset_primary is not None:
    print("=" * 60)
    print("TRAINING: motion_blur_specialist_v3 (EXTREME blur augmentation)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_primary.location}/data.yaml",
        epochs=150,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="motion_blur_specialist_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.9,
        hsv_v=0.6,
        degrees=30,
        translate=0.2,
        scale=0.8,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.3,
        copy_paste=0.2,
        erasing=0.4,
        shear=5.0,
        perspective=0.001,
    )
    print("✅ Training complete: motion_blur_specialist_v3")
else:
    print("⏭️ SKIPPED: motion_blur_specialist_v3 — primary dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🔽 IMMEDIATE DOWNLOAD — motion_blur_specialist_v3
# Run this RIGHT AFTER training completes!
# ════════════════════════════════════════════════════════════════
from google.colab import files
import shutil, os

model_name = "motion_blur_specialist_v3.pt"
run_name = "motion_blur_specialist_v3"
path = f"runs/detect/{run_name}/weights/best.pt"

if os.path.exists(path):
    from ultralytics import YOLO
    metrics = YOLO(path).val()
    map50 = metrics.box.map50
    size_mb = os.path.getsize(path) / 1024 / 1024
    if map50 >= 0.4:
        shutil.copy(path, model_name)
        files.download(model_name)
        print(f"✅ DOWNLOADED: {model_name} mAP50={map50:.3f} ({size_mb:.1f}MB)")
    else:
        print(f"❌ FAILED: {model_name} mAP50={map50:.3f} — below 0.4 specialist threshold, not downloading")
else:
    print(f"❌ MISSING: {path} — training may not have completed")

In [ ]:
# ── Train wide_angle_specialist_v3 ─────────────────────────────
# Reuses primary dataset — epochs=150, imgsz=1280, extreme scale augmentation
# CRITICAL for high school game film (Dustin's St Mark's footage)
if dataset_primary is not None:
    print("=" * 60)
    print("TRAINING: wide_angle_specialist_v3 (imgsz=1280, EXTREME scale)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_primary.location}/data.yaml",
        epochs=150,
        imgsz=1280,
        batch=4,
        name="wide_angle_specialist_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.9,
        hsv_v=0.6,
        degrees=25,
        translate=0.2,
        scale=0.9,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.2,
        copy_paste=0.2,
        erasing=0.4,
        perspective=0.002,
    )
    print("✅ Training complete: wide_angle_specialist_v3")
else:
    print("⏭️ SKIPPED: wide_angle_specialist_v3 — primary dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🔽 IMMEDIATE DOWNLOAD — wide_angle_specialist_v3
# Run this RIGHT AFTER training completes!
# ════════════════════════════════════════════════════════════════
from google.colab import files
import shutil, os

model_name = "wide_angle_specialist_v3.pt"
run_name = "wide_angle_specialist_v3"
path = f"runs/detect/{run_name}/weights/best.pt"

if os.path.exists(path):
    from ultralytics import YOLO
    metrics = YOLO(path).val()
    map50 = metrics.box.map50
    size_mb = os.path.getsize(path) / 1024 / 1024
    if map50 >= 0.4:
        shutil.copy(path, model_name)
        files.download(model_name)
        print(f"✅ DOWNLOADED: {model_name} mAP50={map50:.3f} ({size_mb:.1f}MB)")
    else:
        print(f"❌ FAILED: {model_name} mAP50={map50:.3f} — below 0.4 specialist threshold, not downloading")
else:
    print(f"❌ MISSING: {path} — training may not have completed")

In [ ]:
# ── Train dark_jersey_specialist_v3 ────────────────────────────
# Reuses primary dataset — epochs=150, EXTREME HSV augmentation
# CRITICAL for navy/black jerseys (Dustin's #11 navy jersey)
if dataset_primary is not None:
    print("=" * 60)
    print("TRAINING: dark_jersey_specialist_v3 (EXTREME HSV for navy/black)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_primary.location}/data.yaml",
        epochs=150,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="dark_jersey_specialist_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.03,
        hsv_s=0.9,
        hsv_v=0.8,
        degrees=25,
        translate=0.2,
        scale=0.8,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.2,
        copy_paste=0.2,
        erasing=0.4,
    )
    print("✅ Training complete: dark_jersey_specialist_v3")
else:
    print("⏭️ SKIPPED: dark_jersey_specialist_v3 — primary dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🔽 IMMEDIATE DOWNLOAD — dark_jersey_specialist_v3
# Run this RIGHT AFTER training completes!
# ════════════════════════════════════════════════════════════════
from google.colab import files
import shutil, os

model_name = "dark_jersey_specialist_v3.pt"
run_name = "dark_jersey_specialist_v3"
path = f"runs/detect/{run_name}/weights/best.pt"

if os.path.exists(path):
    from ultralytics import YOLO
    metrics = YOLO(path).val()
    map50 = metrics.box.map50
    size_mb = os.path.getsize(path) / 1024 / 1024
    if map50 >= 0.4:
        shutil.copy(path, model_name)
        files.download(model_name)
        print(f"✅ DOWNLOADED: {model_name} mAP50={map50:.3f} ({size_mb:.1f}MB)")
    else:
        print(f"❌ FAILED: {model_name} mAP50={map50:.3f} — below 0.4 specialist threshold, not downloading")
else:
    print(f"❌ MISSING: {path} — training may not have completed")

In [ ]:
# ── Train partial_visibility_specialist_v3 ───────────────────
# Reuses primary dataset — epochs=150, extreme copy_paste + erasing
if dataset_primary is not None:
    print("=" * 60)
    print("TRAINING: partial_visibility_specialist_v3 (EXTREME erasing)")
    print("=" * 60)

    model = YOLO(V3_BASE)
    model.train(
        data=f"{dataset_primary.location}/data.yaml",
        epochs=150,
        imgsz=V3_IMGSZ,
        batch=V3_BATCH,
        name="partial_visibility_specialist_v3",
        device=V3_DEVICE,
        augment=True,
        hsv_h=0.02,
        hsv_s=0.9,
        hsv_v=0.6,
        degrees=25,
        translate=0.2,
        scale=0.8,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.3,
        copy_paste=0.5,
        erasing=0.6,
    )
    print("✅ Training complete: partial_visibility_specialist_v3")
else:
    print("⏭️ SKIPPED: partial_visibility_specialist_v3 — primary dataset not available")

In [ ]:
# ════════════════════════════════════════════════════════════════
# 🔽 IMMEDIATE DOWNLOAD — partial_visibility_specialist_v3
# Run this RIGHT AFTER training completes!
# ════════════════════════════════════════════════════════════════
from google.colab import files
import shutil, os

model_name = "partial_visibility_specialist_v3.pt"
run_name = "partial_visibility_specialist_v3"
path = f"runs/detect/{run_name}/weights/best.pt"

if os.path.exists(path):
    from ultralytics import YOLO
    metrics = YOLO(path).val()
    map50 = metrics.box.map50
    size_mb = os.path.getsize(path) / 1024 / 1024
    if map50 >= 0.4:
        shutil.copy(path, model_name)
        files.download(model_name)
        print(f"✅ DOWNLOADED: {model_name} mAP50={map50:.3f} ({size_mb:.1f}MB)")
    else:
        print(f"❌ FAILED: {model_name} mAP50={map50:.3f} — below 0.4 specialist threshold, not downloading")
else:
    print(f"❌ MISSING: {path} — training may not have completed")

# ================================================================
# FINAL SUMMARY
# ================================================================

In [ ]:
import os

print("=" * 60)
print("v3 OCR TRAINING COMPLETE \u2014 FINAL REPORT")
print("=" * 60)

all_v3_models = [
    ("jersey_ocr_v3_primary.pt",            "Chunk 1 \u2014 Multi-sport OCR",    0.5),
    ("jersey_ocr_v3_secondary.pt",           "Chunk 1 \u2014 Multi-sport OCR",    0.5),
    ("basketball_ocr_v3.pt",                 "Chunk 2 \u2014 Sport-specific OCR", 0.5),
    ("football_ocr_v3.pt",                   "Chunk 2 \u2014 Sport-specific OCR", 0.5),
    ("lacrosse_ocr_v3.pt",                   "Chunk 2 \u2014 Sport-specific OCR", 0.5),
    ("player_isolator_v3.pt",                "Chunk 3 \u2014 Player isolation",   0.5),
    ("jersey_color_classifier_v3.pt",        "Chunk 3 \u2014 Color classifier",  0.5),
    ("number_region_detector_v3.pt",         "Chunk 3 \u2014 Number region",     0.5),
    ("motion_blur_specialist_v3.pt",         "Chunk 4 \u2014 Motion blur",       0.4),
    ("wide_angle_specialist_v3.pt",          "Chunk 4 \u2014 Wide angle",        0.4),
    ("dark_jersey_specialist_v3.pt",         "Chunk 4 \u2014 Dark jersey",       0.4),
    ("partial_visibility_specialist_v3.pt",  "Chunk 4 \u2014 Partial visibility",0.4),
]

passed = []
failed = []
for name, chunk, threshold in all_v3_models:
    if os.path.exists(name):
        size_mb = os.path.getsize(name) / 1024 / 1024
        try:
            from ultralytics import YOLO
            m = YOLO(name)
            metrics = m.val()
            map50 = metrics.box.map50
            status = "\u2705 PASS" if map50 >= threshold else "\u26a0\ufe0f LOW"
            passed.append(f"  {status}: {name} ({size_mb:.1f}MB, mAP50={map50:.3f}) \u2014 {chunk}")
        except Exception:
            passed.append(f"  \u2705 DOWNLOADED: {name} ({size_mb:.1f}MB, mAP50=unknown) \u2014 {chunk}")
    else:
        run_path = f"runs/detect/{name.replace('.pt', '')}/weights/best.pt"
        if os.path.exists(run_path):
            failed.append(f"  \u26a0\ufe0f TRAINED but NOT DOWNLOADED: {name} \u2014 run its download cell! \u2014 {chunk}")
        else:
            failed.append(f"  \u274c MISSING: {name} \u2014 not trained \u2014 {chunk}")

print(f"\nDOWNLOADED ({len(passed)}/12):")
for m in passed: print(m)

if failed:
    print(f"\nNOT DOWNLOADED ({len(failed)}/12):")
    for m in failed: print(m)
else:
    print("\n\ud83c\udf89 ALL 12 MODELS DOWNLOADED!")

print()
print("\u2500" * 60)
print("FILENAME CROSS-CHECK (must match roboflow_detector.py):")
print("\u2500" * 60)
expected = [
    "jersey_ocr_v3_primary.pt",
    "jersey_ocr_v3_secondary.pt",
    "basketball_ocr_v3.pt",
    "football_ocr_v3.pt",
    "lacrosse_ocr_v3.pt",
    "player_isolator_v3.pt",
    "jersey_color_classifier_v3.pt",
    "number_region_detector_v3.pt",
    "motion_blur_specialist_v3.pt",
    "wide_angle_specialist_v3.pt",
    "dark_jersey_specialist_v3.pt",
    "partial_visibility_specialist_v3.pt",
]
for name in expected:
    exists = "\u2705" if os.path.exists(name) else "\u274c"
    print(f"  {exists} {name}")

print()
print("GIT COMMANDS (run after moving .pt files to app/model/):")
print("  cd playerJerseyIdentification-master")
print("  cp *.pt app/model/")
print('  git add app/model/*.pt')
print('  git commit -m "Add v3 OCR models \u2014 Ali replacement layer"')
print("  git push")
print()
print("VERIFY DEPLOYMENT:")
print("  curl https://jersey-detection-production-d8d8.up.railway.app/health")
print("  Look for roboflow_models_v3_ocr \u2014 all should show 'loaded'")
print()
print("=" * 60)
print(f"v3 OCR PIPELINE: {len(passed)}/12 models ready")
print("=" * 60)

# ================================================================
# AFTER TRAINING — NEXT STEPS
# ================================================================
#
# 1. Move all downloaded .pt files to jersey-detection/app/model/
# 2. Run: git add app/model/*.pt
#         git commit -m "Add v3 OCR models — Ali replacement layer"
#         git push
# 3. Railway auto-deploys in ~3 minutes
# 4. Verify with health check:
#    curl https://jersey-detection-production-d8d8.up.railway.app/health
#    Look for roboflow_models_v3_ocr — all should show "loaded"
# 5. Run the primary test (Dustin's game film):
#    curl -X POST https://jersey-detection-production-d8d8.up.railway.app/analyze \
#      -H "Content-Type: application/json" \
#      -d '{"videoUrl":"https://www.youtube.com/watch?v=XRSrRPbZIF0",
#           "jerseyNumber":11,"jerseyColor":"navy","sport":"football",
#           "position":"QB","timeRangeStart":60,"timeRangeEnd":120}'
#    SUCCESS = detections > 0, dark_jersey_specialist contributed
# 6. If detections still 0 after v3 deploys, send summary to Claude
#    for diagnosis — do NOT retrain without analysis first
#
# DETECTION LAYER ORDER (after v3 deployed):
# Step 1: Ali's ensemble (v1) runs first
# Step 2: jersey_number_universal_v1 (v2, mAP50 0.995)
# Step 3: v3 OCR pipeline (these 12 models)
# Step 4: v2 sport-specific models
# Step 5: temporal_consensus (3+ frame filter)
# Step 6: Stat generation (zones, actions, ball tracking)